In [1]:
import pandas as pd

In [2]:
# Load dataset
df = pd.read_csv("fraud_detection_dirty.csv")

In [3]:
# 1. Missing values
# Check missing values
missing_values = df.isnull().sum()

In [4]:
# Fill numeric columns with median
numeric_cols = df.select_dtypes(include="number").columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

In [5]:
# Fill categorical columns with mode
categorical_cols = df.select_dtypes(include=["object", "string"]).columns

for col in categorical_cols:
    if df[col].notna().any():
        df[col] = df[col].fillna(df[col].mode().iloc[0])

In [6]:
# 2. Remove duplicate rows
df = df.drop_duplicates()

In [7]:
# Clean column names
df.columns = df.columns.str.strip()

# Display all column names
print(df.columns.tolist())

['transaction_id', 'time_seconds', 'amount_inr', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'merchant_category', 'card_type', 'entry_mode', 'city_tier', 'is_foreign', 'customer_age_group', 'account_age_months', 'hour_of_day', 'Class', 'notes', 'reviewed_by']


In [8]:
# 3. Clean impossible transaction amounts
# Keep only positive transaction amounts
df["amount_inr"] = pd.to_numeric(df["amount_inr"], errors="coerce")
df.loc[df["amount_inr"] <= 0, "amount_inr"] = pd.NA

# Replace invalid amounts with median
df["amount_inr"] = df["amount_inr"].fillna(df["amount_inr"].median())

In [9]:
# 4. Standardize fraud/class labels

df["Class"] = (
    df["Class"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "fraud": 1,
        "yes": 1,
        "1.0": 1,
        "1": 1,
        "not fraud": 0,
        "no": 0,
        "0.0": 0,
        "0": 0
    })
)

df["Class"] = pd.to_numeric(
    df["Class"],
    errors="coerce"
).astype("Int64")

In [10]:
# 5. Standardize merchant category
df["merchant_category"] = (
    df["merchant_category"]
    .astype("string")
    .str.strip()
    .str.lower()
)

df["merchant_category"] = df["merchant_category"].replace({
    "retail": "retail",
    "atm": "atm"
})

In [11]:
# 6. Standardize entry mode
df["entry_mode"] = (
    df["entry_mode"]
    .astype("string")
    .str.strip()
    .str.lower()
)

In [12]:
# 7. Validate transaction time

df["time_seconds"] = pd.to_numeric(
    df["time_seconds"],
    errors="coerce"
)

# Valid range: 0–86400 seconds
df.loc[
    ~df["time_seconds"].between(0, 86400),
    "time_seconds"
] = pd.NA

In [13]:
# 8. Remove whitespace from transaction_id
df["transaction_id"] = (
    df["transaction_id"]
    .astype("string")
    .str.strip()
)

In [14]:
# 9. Convert is_foreign to Boolean
df["is_foreign"] = (
    df["is_foreign"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "yes": True,
        "true": True,
        "1": True,
        "no": False,
        "false": False,
        "0": False
    })
)

In [15]:
# 10. Drop completely blank columns
df = df.dropna(axis=1, how="all")

In [16]:
# Final validation
print("Dataset Shape:", df.shape)
print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)

print("\nCleaned Dataset:")
print(df.head())

Dataset Shape: (10004, 40)

Missing Values:
transaction_id         0
time_seconds          20
amount_inr             0
V1                     0
V2                     0
V3                     0
V4                     0
V5                     0
V6                     0
V7                     0
V8                     0
V9                     0
V10                    0
V11                    0
V12                    0
V13                    0
V14                    0
V15                    0
V16                    0
V17                    0
V18                    0
V19                    0
V20                    0
V21                    0
V22                    0
V23                    0
V24                    0
V25                    0
V26                    0
V27                    0
V28                    0
merchant_category      0
card_type              0
entry_mode             0
city_tier              0
is_foreign             0
customer_age_group     0
account_age_months     0
hour_o

In [17]:
# 1. Remove duplicate rows
df = df.drop_duplicates().copy()

In [18]:
# 2. Handle missing Class values
# Class is the fraud target, so don't guess fraud/non-fraud.
# Remove rows where the target is missing.
df = df.dropna(subset=["Class"]).copy()

In [19]:
# 3. Keep invalid time values as missing
# Valid range: 0–86400 seconds
df.loc[
    ~df["time_seconds"].between(0, 86400),
    "time_seconds"
] = pd.NA

In [20]:
# 4. Standardize transaction ID
df["transaction_id"] = (
    df["transaction_id"]
    .astype("string")
    .str.strip()
)

In [21]:
# 5. Final duplicate check
duplicate_count = df.duplicated().sum()

In [22]:
# 6. Final missing-value check
missing_values = df.isnull().sum()

In [23]:
# 7. Final validation
print("====================================")
print("FINAL DATASET VALIDATION")
print("====================================")

print("\nDataset Shape:")
print(df.shape)

print("\nDuplicate Rows:")
print(duplicate_count)

print("\nMissing Values:")
print(missing_values[missing_values > 0])

print("\nClass Distribution:")
print(df["Class"].value_counts(dropna=False))

print("\nData Types:")
print(df.dtypes)

print("\nFirst 5 Rows:")
print(df.head())

FINAL DATASET VALIDATION

Dataset Shape:
(9994, 40)

Duplicate Rows:
0

Missing Values:
time_seconds    20
dtype: int64

Class Distribution:
Class
0    9772
1     222
Name: count, dtype: Int64

Data Types:
transaction_id         string
time_seconds          float64
amount_inr            float64
V1                    float64
V2                    float64
V3                    float64
V4                    float64
V5                    float64
V6                    float64
V7                    float64
V8                    float64
V9                    float64
V10                   float64
V11                   float64
V12                   float64
V13                   float64
V14                   float64
V15                   float64
V16                   float64
V17                   float64
V18                   float64
V19                   float64
V20                   float64
V21                   float64
V22                   float64
V23                   float64
V24           

In [27]:
# -----------------------------------------
# 1. Clean merchant category
# -----------------------------------------

df["merchant_category"] = (
    df["merchant_category"]
    .astype(str)
    .str.strip()
    .str.lower()
)

category_mapping = {
    "atm": "atm withdrawal",
    "food": "food & dining"
}

df["merchant_category"] = (
    df["merchant_category"]
    .replace(category_mapping)
)


# -----------------------------------------
# 2. Clean fraud label
# -----------------------------------------

fraud_mapping = {
    "fraud": 1,
    "yes": 1,
    "1": 1,
    "1.0": 1,
    "true": 1,
    "non-fraud": 0,
    "no": 0,
    "0": 0,
    "0.0": 0,
    "false": 0
}

df["Class"] = (
    df["Class"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(fraud_mapping)
)

df["class"] = pd.to_numeric(
    df["Class"],
    errors="coerce"
)


# -----------------------------------------
# 3. Fraud analysis by merchant category
# -----------------------------------------

fraud_by_category = (
    df.groupby("merchant_category")
      .agg(
          total_transactions=("transaction_id", "count"),
          fraudulent_transactions=("Class", "sum")
      )
      .reset_index()
)


# -----------------------------------------
# 4. Calculate fraud rate
# -----------------------------------------

fraud_by_category["fraud_rate"] = (
    fraud_by_category["fraudulent_transactions"]
    / fraud_by_category["total_transactions"]
    * 100
)


# -----------------------------------------
# 5. Sort highest fraud rate first
# -----------------------------------------

fraud_by_category = fraud_by_category.sort_values(
    "fraud_rate",
    ascending=False
)


# -----------------------------------------
# 6. Round fraud rate
# -----------------------------------------

fraud_by_category["fraud_rate"] = (
    fraud_by_category["fraud_rate"].round(2)
)

fraud_by_category

,merchant_category,total_transactions,fraudulent_transactions,fraud_rate
0,atm withdrawal,978,72,7.36
6,online shopping,956,61,6.38
1,electronics,1708,78,4.57
4,grocery,890,3,0.34
3,fuel,918,2,0.22
7,retail,922,2,0.22
9,utilities,931,2,0.21
5,healthcare,878,1,0.11
2,food & dining,900,1,0.11
8,travel,913,0,0.0


In [28]:
# Save cleaned dataset
df.to_csv("credit_card_fraud_cleaned.csv", index=False)

print("Cleaned dataset saved successfully.")

Cleaned dataset saved successfully.
